# Tutorial 4 — Reconstruction

**Goal:** reconstruct volumes from sinograms, compare algorithms and filters, and measure
how faithful the result is against the phantom.

**You will learn:** `reconstruct`, `reconstruct_pair`, `AVAILABLE_ALGORITHMS`, and the
difference between analytic and iterative methods.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

## 1. Which algorithms can I use?

**FBP** (filtered back-projection) runs everywhere. The iterative algorithms (SIRT, SART,
CGLS, …) need the ASTRA toolbox and a CUDA GPU; without it DIANA warns and falls back to FBP,
so the notebook still runs.

In [ ]:
from neutron_xray_sim.acquisition.projector import ASTRA_OK
print("available algorithms:", nxs.AVAILABLE_ALGORITHMS)
print("ASTRA GPU backend   :", "yes" if ASTRA_OK else "no (CPU fallbacks will be used)")

In [ ]:
phantom = nxs.make_phantom("composite", N=64)
xray, neutron = nxs.make_sinogram_pair(phantom, n_angles=120, verbose=False)
vol_x, vol_n = nxs.reconstruct_pair(xray, neutron, algorithm="FBP", verbose=False)
print(vol_x.shape, "cm⁻¹")

## 2. Reconstruction vs ground truth

Reconstructed values are in physical units (cm⁻¹), so they can be compared voxel by voxel
with the phantom. The X-ray ground truth is taken at 80 keV, a typical effective energy of the
120 kVp spectrum — differences therefore also include **beam hardening**.

In [ ]:
s = 32
gt_x = phantom.mu_x_vols[nxs.DEFAULT_GT_ENERGY_IDX]
gt_n = phantom.mu_n_vol
fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for row, (gt, rec, name) in enumerate([(gt_x, vol_x, "X-ray"), (gt_n, vol_n, "neutron")]):
    vmax = np.percentile(gt, 99.5)
    for ax, img, t in zip(axes[row], [gt[s], rec[s], rec[s] - gt[s]],
                          ["phantom", "FBP reconstruction", "difference"]):
        kw = dict(cmap="RdBu_r", vmin=-vmax / 2, vmax=vmax / 2) if t == "difference" \
             else dict(cmap="gray", vmin=0, vmax=vmax)
        im = ax.imshow(img, **kw); ax.set_title(f"{name}: {t}"); ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

## 3. Filters and number of projections

FBP needs a ramp filter; smoother windows (Hann, Hamming) trade resolution for noise.
With few projections, streaks appear (angular undersampling).

In [ ]:
def rmse(a, b, mask):
    return float(np.sqrt(np.mean((a[mask] - b[mask]) ** 2)))

inside = phantom.label_vol > 0
cfg = nxs.ArtifactConfig(photon_noise=True, I0_xray=1e4, I0_neutron=1e4)
_, noisy = nxs.inject_sinogram_artifacts(xray, neutron, cfg)
for filt in ["ram-lak", "shepp-logan", "hann"]:
    v = nxs.reconstruct(noisy, algorithm="FBP", filter_name=filt)
    print(f"{filt:>12}: neutron RMSE = {rmse(v, gt_n, inside):.3f} cm⁻¹")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, n_ang in zip(axes, [15, 30, 60, 180]):
    _, n_sino = nxs.make_sinogram_pair(phantom, n_angles=n_ang, verbose=False)
    v = nxs.reconstruct(n_sino)
    ax.imshow(v[s], cmap="gray", vmin=0, vmax=2.5); ax.axis("off")
    ax.set_title(f"{n_ang} angles — RMSE {rmse(v, gt_n, inside):.2f}")

## 4. Iterative algorithms (GPU)

With ASTRA installed, try `algorithm="SIRT"` (with `n_iter=100`), `"SART"`, `"CGLS"` or
`"TV_MIN"` (total-variation regularised). They cope much better with few or noisy projections.
On a CPU-only machine these calls fall back to FBP with a warning.

In [ ]:
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    v_sirt = nxs.reconstruct(noisy, algorithm="SIRT", n_iter=100)
for w in caught:
    print("warning:", w.message)
print(f"SIRT neutron RMSE = {rmse(v_sirt, gt_n, inside):.3f} cm⁻¹")

**Exercise:** reconstruct with 30 noisy projections using FBP. Then (if you have a GPU)
compare with SIRT and TV_MIN. Which algorithm keeps the water inclusion separable from HDPE?